# Exact WorM VSR at the viable boundary: pretrained GRU-1024 hybrid handover

This is a **new prospective experiment** defined after the independent viability result:

- official WorM GRU-1024:
  - load 2 ≈ 0.99
  - load 3 ≈ 0.97
  - load 4 ≈ 0.939

Therefore the operating point is fixed at **VSR load 3**, where the recurrent route independently passes the pre-specified `0.95` viability threshold.

## Goal

Construct a recurrent–retrieval hybrid around the **official pretrained WorM GRU-1024 backbone**, establish that the added retrieval route is also sufficient at load 3, create a retrieval-dominant converged hybrid using only model-side criteria, and then test whether a post-convergence retrieval price causes causal handover back to recurrence.

The notebook stops if a prerequisite fails.

## Important interpretation rule

This is not a reproduction of a spontaneously retrieval-dominant free hybrid. It is a **targeted exact-task causal validation** on a human benchmark task, using a pretrained recurrent backbone whose independent viability was established before this experiment.

Human curves are only introduced after the model-side operating point and steering hyperparameters are fixed.

## What was fixed relative to the previous run

The previous hybrid reported `recurrent = 0.13`, `retrieval = 0.54`. Neither
number was interpretable. The two causes were:

1. **The backbone forward pass was rebuilt by hand** (CNN -> projection ->
   task embedding -> GRU, with hardcoded widths `512` / `498`, a guessed task
   id `6`, and no `seq_len`). The hybrid now *calls*
   `self.backbone(stim, "VSR_Task", seq_len)` and captures the GRU states from
   that same forward pass with a temporary hook.
2. **The "frozen" backbone was put in `.train()` mode**, so normalisation
   running statistics kept updating even though `requires_grad=False`. The
   backbone is now held in `eval()` during retrieval-only training, and cell 11
   asserts that recurrent accuracy is bit-identical before and after.

A strict sanity gate (cell 8b) now blocks everything downstream until
`official logits == hybrid recurrent logits` and `official hidden == hybrid
hidden` exactly, and hybrid recurrent L3 accuracy reproduces the official
0.970.

In [ ]:
# 1) Imports and paths
import os, sys, math, random, time, copy, json, subprocess, zipfile
from pathlib import Path
from argparse import Namespace

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Subset
from scipy import stats

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", DEVICE)

try:
    from google.colab import drive
    drive.mount("/content/drive")
    RESULT_DIR = Path(os.environ.get("RESULTS_DIR", "results/ICLR2027_WorM_L3_handover"))
except Exception:
    RESULT_DIR = Path("/content/ICLR2027_WorM_L3_handover")

RESULT_DIR.mkdir(parents=True, exist_ok=True)
print("results:", RESULT_DIR)

In [ ]:
# 2) Clone pinned official WorM repository
REPO_DIR = Path("/content/WorM")
WORM_COMMIT = "84c867c89bacc2921edb41716540a246a4fe061b"

if not REPO_DIR.exists():
    subprocess.run(
        ["git", "clone", "https://github.com/ZhangLab-DeepNeuroCogLab/WorM.git", str(REPO_DIR)],
        check=True,
    )

subprocess.run(["git", "-C", str(REPO_DIR), "fetch", "--all"], check=True)
subprocess.run(["git", "-C", str(REPO_DIR), "checkout", WORM_COMMIT], check=True)

if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))

from src.data.dataset import Visual_Serial_Recall_Recognition_Dataset
from src.model import WM_Model

print("WorM commit:", WORM_COMMIT)

In [ ]:
# 3) Configuration
class CFG:
    max_seq_len = 20
    img_size = 96
    rs_img_size = 32
    grid_size = 6

    # NOTE: n_classes is only a *fallback*. The real value is read from the
    # official vsr_task_head in cell 7b and asserted against the labels.
    n_classes_fallback = 9

    # Prospectively fixed from independent official-model viability result.
    operating_load = 3
    viability_threshold = 0.95
    clean_threshold = 0.95

    # Train the new retrieval route only in the locally viable region.
    retrieval_train_loads = (2, 3)

    # Later diagnostics only.
    human_loads = (3, 4, 5, 6)

    # Exact official generator; practical VSR-only subset for Colab.
    num_samples = 16000
    batch_size = 48
    num_workers = 2

    # Retrieval route
    attn_dim = 256
    retrieval_steps = 5000
    retrieval_lr = 3e-4

    # Cache frozen-backbone hidden states for the retrieval-only phase.
    # The backbone does not change during that phase, so this is exact and
    # roughly two orders of magnitude cheaper than re-running the CNN+GRU.
    cache_backbone_features = True

    # Keep BatchNorm running statistics frozen everywhere. Without this the
    # "frozen" backbone silently drifts in .train() mode even with
    # requires_grad=False, which is one way to destroy the 0.97 recurrent route.
    freeze_norm_stats = True

    # Model-only calibration before human curves.
    beta_grid = (1.0, 2.0, 4.0, 8.0)

    # Model-only steering pilot.
    lambda_grid = (1e-4, 3e-4, 1e-3, 3e-3, 1e-2)
    pilot_steps = 2000

    # Confirmatory paired seeds.
    seeds = (0, 1, 2, 3, 4)
    steer_steps = 4000
    withdrawal_steps = 2000
    lr_steer = 1e-4

    trajectory_every = 500
    grad_clip = 1.0

    # Fixed resampling permutation so Dh / Dc are comparable across arms.
    perm_seed = 7

print({k: v for k, v in CFG.__dict__.items()
       if not k.startswith("_") and not callable(v)})

In [ ]:
# 4) Generate/load exact official VSR dataset
DATA_DIR = Path("/content/worm_vsr_l3_exact")
DATA_DIR.mkdir(parents=True, exist_ok=True)

random.seed(12345)
np.random.seed(12345)
torch.manual_seed(12345)

train_full = Visual_Serial_Recall_Recognition_Dataset(
    data_path=str(DATA_DIR),
    max_seq_len=CFG.max_seq_len,
    probe_variant="Recall",
    grid_size=CFG.grid_size,
    list_length_options=[2,3,4,5,6,7,8,9],
    distractor_difference_options=[],
    gen_random_trials=True,
    held_out_list_lengths=[],
    held_out_distractor_diffs=[],
    num_samples=CFG.num_samples,
    img_size=CFG.img_size,
    rs_img_size=CFG.rs_img_size,
    write=True,
    split="train",
)

test_full = Visual_Serial_Recall_Recognition_Dataset(
    data_path=str(DATA_DIR),
    max_seq_len=CFG.max_seq_len,
    probe_variant="Recall",
    rs_img_size=CFG.rs_img_size,
    split="test",
)

def indices_for_loads(dataset, loads):
    loads = set(int(x) for x in loads)
    return [
        i for i, meta in enumerate(dataset.data_dict)
        if int(meta["list_length"]) in loads
    ]

train_idx = indices_for_loads(train_full, CFG.retrieval_train_loads)
train_loader = DataLoader(
    Subset(train_full, train_idx),
    batch_size=CFG.batch_size,
    shuffle=True,
    drop_last=True,
    num_workers=CFG.num_workers,
    pin_memory=torch.cuda.is_available(),
)

test_loaders = {}
for L in sorted(set(CFG.human_loads) | {2}):
    idx = indices_for_loads(test_full, [L])
    test_loaders[L] = DataLoader(
        Subset(test_full, idx),
        batch_size=CFG.batch_size,
        shuffle=False,
        drop_last=False,
        num_workers=CFG.num_workers,
        pin_memory=torch.cuda.is_available(),
    )
    print(f"test load {L}: {len(idx)} trials")

print("retrieval-train trials:", len(train_idx))

In [ ]:
# 5) Download and load official pretrained GRU-1024
PRETRAIN_DIR = Path("/content/worm_official_pretrained")
PRETRAIN_DIR.mkdir(parents=True, exist_ok=True)
ZIP_PATH = PRETRAIN_DIR / "Trained_Models.zip"
EXTRACT_DIR = PRETRAIN_DIR / "extracted"
MODEL_PATH = EXTRACT_DIR / "Trained_Models/GRU/GRU_1024/model.pt"

if not MODEL_PATH.exists():
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "gdown"], check=True)
    import gdown

    if not ZIP_PATH.exists():
        gdown.download(
            id="1kcs6-r247XxGIb-06_09Fc3wHgK0Tysf",
            output=str(ZIP_PATH),
            quiet=False,
        )

    EXTRACT_DIR.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(ZIP_PATH) as z:
        z.extractall(EXTRACT_DIR)

ckpt = torch.load(MODEL_PATH, map_location="cpu", weights_only=False)
config_obj = ckpt["config"]
config_dict = vars(config_obj) if not isinstance(config_obj, dict) else config_obj
state_dict = ckpt.get("model_state_dict", ckpt.get("state_dict"))

rename = {
    "stsc_task_head.": "sts_task_head.",
    "virec_2c_task_head.": "vir_task_head.",
    "complex_wm_task_head.": "cs_task_head.",
}
patched = {}
for k, v in state_dict.items():
    nk = k
    for old, new in rename.items():
        if k.startswith(old):
            nk = new + k[len(old):]
            break
    patched[nk] = v

official_cfg = Namespace(**config_dict)
official_model = WM_Model(official_cfg, DEVICE).to(DEVICE)
official_model.load_state_dict(patched, strict=True)
official_model.eval()

print("official GRU-1024 loaded")
print("parameters:", round(sum(p.numel() for p in official_model.parameters()) / 1e6, 2), "M")

In [ ]:
# 6) Batch helpers
def move_batch(batch):
    stim, resp, seq_len, list_length = batch
    return (
        stim.to(DEVICE, non_blocking=True),
        resp.to(DEVICE, non_blocking=True),
        torch.as_tensor(seq_len, device=DEVICE).long(),
        torch.as_tensor(list_length, device=DEVICE).long(),
    )

def response_indices(seq_len, list_length):
    # Only valid when every trial in the batch has the same list length,
    # which is true for the per-load test loaders. Assert it instead of
    # silently indexing with the first trial's length.
    assert int(list_length.min()) == int(list_length.max()), (
        "response_indices() requires a single-load batch; use "
        "response_mask_general() for mixed-load batches."
    )
    L = int(list_length[0].item())
    offsets = torch.arange(L, device=seq_len.device)[None, :]
    return (seq_len - list_length)[:, None] + offsets

def infinite_loader(loader):
    while True:
        for batch in loader:
            yield batch

def response_mask_general(seq_len, list_length, T):
    t = torch.arange(T, device=seq_len.device)[None, :]
    start = (seq_len - list_length)[:, None]
    end = seq_len[:, None]
    return (t >= start) & (t < end)

def make_train_loader(seed):
    """Seeded shuffling, so matched arms see matched data order."""
    g = torch.Generator()
    g.manual_seed(int(seed))
    return DataLoader(
        Subset(train_full, train_idx),
        batch_size=CFG.batch_size,
        shuffle=True,
        drop_last=True,
        num_workers=CFG.num_workers,
        pin_memory=torch.cuda.is_available(),
        generator=g,
    )

def freeze_norm_(module):
    """Put every normalisation layer with running stats into eval mode."""
    for m in module.modules():
        if isinstance(m, (nn.BatchNorm1d, nn.BatchNorm2d, nn.BatchNorm3d,
                          nn.SyncBatchNorm, nn.InstanceNorm1d,
                          nn.InstanceNorm2d, nn.InstanceNorm3d)):
            m.eval()

In [ ]:
# 7) Verify recurrent operating point
@torch.no_grad()
def official_recurrent_accuracy(model, load):
    model.eval()
    correct = total = 0

    for batch in test_loaders[load]:
        stim, resp, seq_len, list_length = move_batch(batch)
        out, _, _, _, _ = model(stim, "VSR_Task", seq_len)

        B = stim.shape[0]
        idx = response_indices(seq_len, list_length)
        bidx = torch.arange(B, device=DEVICE)[:, None]

        logits = out[bidx, idx]
        y = resp[bidx, idx]

        correct += (logits.argmax(-1) == y).sum().item()
        total += y.numel()

    return correct / total

for L in [2,3,4,5,6]:
    print(f"official recurrent load {L}: {official_recurrent_accuracy(official_model, L):.4f}")

operating_acc = official_recurrent_accuracy(official_model, CFG.operating_load)
assert operating_acc >= CFG.viability_threshold, (
    f"STOP: official recurrent route is {operating_acc:.4f} at load "
    f"{CFG.operating_load}, below threshold {CFG.viability_threshold}."
)

print("PASS: recurrent route viable at operating load", CFG.operating_load)

In [ ]:
# 7b) Probe the EXACT official interface (no hardcoded shapes)
#
# Everything the hybrid needs -- hidden size, number of classes, tensor layout
# -- is read off the official model here instead of being guessed.

TASK_NAME = "VSR_Task"

def _unwrap_rnn_output(o):
    if isinstance(o, tuple):
        o = o[0]
    if isinstance(o, nn.utils.rnn.PackedSequence):
        o, _ = nn.utils.rnn.pad_packed_sequence(o, batch_first=True)
    return o

class _GRUCapture:
    """Temporary forward hook on the official GRU module."""
    def __init__(self, gru_module):
        self.mod = gru_module
        self.h = None
        self.handle = None
    def __enter__(self):
        def hook(m, inp, out):
            self.h = _unwrap_rnn_output(out)
        self.handle = self.mod.register_forward_hook(hook)
        return self
    def __exit__(self, *exc):
        self.handle.remove()
        self.handle = None
        return False

def _last_linear_out_features(module):
    last = None
    for m in module.modules():
        if isinstance(m, nn.Linear):
            last = m
    if last is None:
        raise RuntimeError("no nn.Linear found in the VSR head")
    return last.out_features

# one probe batch at the operating load
_probe_batch = next(iter(test_loaders[CFG.operating_load]))
_stim, _resp, _seq_len, _ll = move_batch(_probe_batch)

official_model.eval()
with torch.no_grad(), _GRUCapture(official_model.MEM) as cap:
    _out = official_model(_stim, TASK_NAME, _seq_len)
    _logits = _out[0] if isinstance(_out, tuple) else _out
    _h = cap.h

B_probe, T_probe = _stim.shape[0], _stim.shape[1]

# GRU output layout: (B, T, D) or (T, B, D)?
GRU_BATCH_FIRST = (_h.shape[0] == B_probe and _h.shape[1] <= T_probe)
if not GRU_BATCH_FIRST:
    assert _h.shape[1] == B_probe, f"unexpected GRU output shape {tuple(_h.shape)}"

HIDDEN_SIZE = int(_h.shape[-1])
N_CLASSES = int(_last_linear_out_features(official_model.vsr_task_head))

print("stim          :", tuple(_stim.shape))
print("official out  :", tuple(_logits.shape))
print("GRU states    :", tuple(_h.shape), "| batch_first =", GRU_BATCH_FIRST)
print("HIDDEN_SIZE   :", HIDDEN_SIZE)
print("N_CLASSES     :", N_CLASSES)
print("max label     :", int(_resp.max()))

assert _logits.shape[0] == B_probe and _logits.shape[1] == T_probe
assert _logits.shape[-1] == N_CLASSES, (
    f"VSR head out_features {N_CLASSES} != official logits {_logits.shape[-1]}"
)
assert int(_resp.max()) < N_CLASSES, "labels exceed the head's output size"
if N_CLASSES != CFG.n_classes_fallback:
    print(f"NOTE: N_CLASSES={N_CLASSES}, not the assumed "
          f"{CFG.n_classes_fallback}. Using the official value.")

CHANCE = 1.0 / N_CLASSES
print("CHANCE        :", round(CHANCE, 4))

def official_hidden_states(stim, seq_len, model=None):
    """Exact official GRU states, aligned to (B, T, D)."""
    model = official_model if model is None else model
    with _GRUCapture(model.MEM) as cap:
        out = model(stim, TASK_NAME, seq_len)
    logits = out[0] if isinstance(out, tuple) else out
    h = cap.h
    if not GRU_BATCH_FIRST:
        h = h.transpose(0, 1)
    T = stim.shape[1]
    if h.shape[1] < T:                       # packed sequences pad to batch max
        pad = torch.zeros(h.shape[0], T - h.shape[1], h.shape[2],
                          dtype=h.dtype, device=h.device)
        h = torch.cat([h, pad], dim=1)
    return logits, h

del _probe_batch, _stim, _resp, _seq_len, _ll, _out, _logits, _h

## Hybrid construction

The official pretrained CNN, projection, task embedding, GRU, and recurrent VSR head are copied exactly.

The new retrieval route:
- queries with the current GRU state;
- keys/values are earlier GRU states only;
- has its own output head;
- combines additively with the official recurrent logits.

A fixed scalar `beta` only calibrates the relative logit scale of the two routes. It is selected using **model-only causal criteria before human curves are loaded**, then frozen for all arms.

In [ ]:
# 8) Hybrid wrapper -- the official forward pass is CALLED, never rebuilt
#
# Previous version reconstructed CNN -> projection -> task embedding -> GRU by
# hand, with hardcoded feature widths (512, 498) and a guessed task id (6), and
# never passed seq_len. That is what produced recurrent = 0.13 instead of 0.97.
#
# Here the hybrid OWNS a deepcopy of the official model and calls
#     self.backbone(stim, "VSR_Task", seq_len)
# so the recurrent route IS the official VSR output, bit for bit, while still
# being trainable later during joint steering. The GRU states are captured with
# a temporary forward hook on that same forward pass.

class PretrainedWorMHybrid(nn.Module):
    def __init__(self, official, attn_dim=256, beta=1.0):
        super().__init__()
        self.backbone = copy.deepcopy(official)

        self.hidden_size = HIDDEN_SIZE
        self.n_classes = N_CLASSES

        # buffer, so beta travels with state_dict / deepcopy; use set_beta()
        self.register_buffer("beta", torch.tensor(float(beta)))

        self.q_proj = nn.Linear(self.hidden_size, attn_dim, bias=False)
        self.k_proj = nn.Linear(self.hidden_size, attn_dim, bias=False)
        self.v_proj = nn.Linear(self.hidden_size, attn_dim, bias=False)

        self.context_mlp = nn.Sequential(
            nn.Linear(attn_dim, attn_dim),
            nn.GELU(),
        )
        self.context_head = nn.Linear(attn_dim, self.n_classes)

        # zero init => at step 0 the hybrid output equals the official output
        nn.init.zeros_(self.context_head.weight)
        nn.init.zeros_(self.context_head.bias)

    def set_beta(self, value):
        with torch.no_grad():
            self.beta.fill_(float(value))
        return self

    @property
    def beta_value(self):
        return float(self.beta.item())

    def retrieval_from_h(self, h, seq_len):
        """Query h_t, keys/values h_{<t}, padding-masked. No in-place ops."""
        B, T, D = h.shape

        q = self.q_proj(h)
        k = self.k_proj(h)
        v = self.v_proj(h)

        scores = torch.matmul(q, k.transpose(-1, -2)) / math.sqrt(q.shape[-1])

        # strictly earlier timesteps only
        causal = torch.ones(T, T, dtype=torch.bool, device=h.device).tril(-1)
        # and never attend to padded timesteps (t >= seq_len)
        valid_key = (torch.arange(T, device=h.device)[None, :] < seq_len[:, None])
        allowed = causal[None, :, :] & valid_key[:, None, :]

        neg = torch.finfo(scores.dtype).min
        scores = scores.masked_fill(~allowed, neg)
        attn = scores.softmax(dim=-1)

        # rows with no allowed key (t = 0) get a uniform softmax; kill them
        # out-of-place -- the old attn[:, 0, :] = 0.0 broke softmax's backward.
        has_key = allowed.any(dim=-1, keepdim=True).to(attn.dtype)
        attn = attn * has_key

        c = torch.matmul(attn, v)
        logits_c_raw = self.context_head(self.context_mlp(c))
        logits_c = self.beta * logits_c_raw
        return c, logits_c, logits_c_raw, attn

    def forward(self, stim, seq_len):
        logits_h, h = official_hidden_states(stim, seq_len, model=self.backbone)
        c, logits_c, logits_c_raw, attn = self.retrieval_from_h(h, seq_len)
        return {
            "h": h,
            "c": c,
            "logits_h": logits_h,
            "logits_c": logits_c,
            "logits_c_raw": logits_c_raw,
            "logits": logits_h + logits_c,
            "attn": attn,
        }

def build_hybrid(seed, beta=1.0):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

    m = PretrainedWorMHybrid(
        official_model,
        attn_dim=CFG.attn_dim,
        beta=beta,
    ).to(DEVICE)
    m.eval()
    return m

In [ ]:
# 9) Retrieval-only training (backbone frozen AND in eval mode)
def set_retrieval_only_trainable(model):
    for p in model.parameters():
        p.requires_grad = False
    for module in [model.q_proj, model.k_proj, model.v_proj,
                   model.context_mlp, model.context_head]:
        for p in module.parameters():
            p.requires_grad = True

def retrieval_modules(model):
    return [model.q_proj, model.k_proj, model.v_proj,
            model.context_mlp, model.context_head]

def train_mode_retrieval_only(model):
    """Retrieval route in train mode, backbone strictly in eval mode.

    requires_grad=False does NOT freeze BatchNorm running statistics. The old
    blanket model.train() let the backbone's norm layers drift on every step,
    which quietly destroys the pretrained recurrent route.
    """
    model.eval()
    for m in retrieval_modules(model):
        m.train()

# ---- optional exact feature cache (backbone is frozen in this phase) --------
FEATURE_CACHE = None

@torch.no_grad()
def build_feature_cache():
    official_model.eval()
    loader = DataLoader(
        Subset(train_full, train_idx),
        batch_size=CFG.batch_size,
        shuffle=False,
        drop_last=False,
        num_workers=CFG.num_workers,
        pin_memory=torch.cuda.is_available(),
    )
    H, R, M, S = [], [], [], []
    for batch in loader:
        stim, resp, seq_len, list_length = move_batch(batch)
        _, h = official_hidden_states(stim, seq_len, model=official_model)
        mask = response_mask_general(seq_len, list_length, resp.shape[1])
        H.append(h.half().cpu())
        R.append(resp.cpu())
        M.append(mask.cpu())
        S.append(seq_len.cpu())
    cache = (torch.cat(H), torch.cat(R), torch.cat(M), torch.cat(S))
    print("feature cache:", tuple(cache[0].shape),
          f"({cache[0].numel()*2/1e6:.0f} MB fp16)")
    return cache

if CFG.cache_backbone_features and FEATURE_CACHE is None:
    FEATURE_CACHE = build_feature_cache()

def train_retrieval_only(model, steps, seed):
    set_retrieval_only_trainable(model)
    trainable = [p for p in model.parameters() if p.requires_grad]
    opt = torch.optim.AdamW(trainable, lr=CFG.retrieval_lr, weight_decay=0.0)

    g = torch.Generator()
    g.manual_seed(int(seed))

    if FEATURE_CACHE is not None:
        Hc, Rc, Mc, Sc = FEATURE_CACHE
        N = Hc.shape[0]
    else:
        it = infinite_loader(make_train_loader(seed))

    for step in range(1, steps + 1):
        train_mode_retrieval_only(model)

        if FEATURE_CACHE is not None:
            sel = torch.randint(0, N, (CFG.batch_size,), generator=g)
            h = Hc[sel].to(DEVICE).float()
            resp = Rc[sel].to(DEVICE)
            mask = Mc[sel].to(DEVICE)
            seq_len = Sc[sel].to(DEVICE)
            _, _, logits_c_raw, _ = model.retrieval_from_h(h, seq_len)
        else:
            stim, resp, seq_len, list_length = move_batch(next(it))
            out = model(stim, seq_len)
            mask = response_mask_general(seq_len, list_length, resp.shape[1])
            logits_c_raw = out["logits_c_raw"]

        loss = F.cross_entropy(logits_c_raw[mask], resp[mask])

        opt.zero_grad(set_to_none=True)
        loss.backward()
        nn.utils.clip_grad_norm_(trainable, CFG.grad_clip)
        opt.step()

        if step % 1000 == 0:
            print(f"retrieval step {step}/{steps} loss={loss.item():.4f}")

    model.eval()

In [ ]:
# 10) Route accuracy and causal metrics
@torch.no_grad()
def route_accuracy(model, load, route="full"):
    was_training = model.training
    model.eval()
    correct = total = 0
    try:
        for batch in test_loaders[load]:
            stim, resp, seq_len, list_length = move_batch(batch)
            out = model(stim, seq_len)

            B = stim.shape[0]
            idx = response_indices(seq_len, list_length)
            bidx = torch.arange(B, device=DEVICE)[:, None]

            key = {"full": "logits", "recurrent": "logits_h",
                   "retrieval": "logits_c", "retrieval_raw": "logits_c_raw"}[route]
            logits = out[key][bidx, idx]
            y = resp[bidx, idx]

            correct += (logits.argmax(-1) == y).sum().item()
            total += y.numel()
    finally:
        if was_training:
            model.train()
    return correct / total

def _derangement(B, generator, device):
    """Random permutation with no fixed points (a trial never resamples itself)."""
    if B < 2:
        return torch.arange(B, device=device)
    p = torch.randperm(B, generator=generator)
    fixed = (p == torch.arange(B)).nonzero(as_tuple=True)[0]
    for i in fixed.tolist():
        j = (i + 1) % B
        p[i], p[j] = p[j].clone(), p[i].clone()
    return p.to(device)

@torch.no_grad()
def causal_metrics(model, load, perm_seed=None):
    """Dr = clip[(a - a_resampled) / (a - 1/K)].

    Permutation is seeded so Dh / Dc are comparable across arms and steps.
    """
    was_training = model.training
    model.eval()
    g = torch.Generator()
    g.manual_seed(CFG.perm_seed if perm_seed is None else int(perm_seed))

    clean_correct = hp_correct = cp_correct = total = 0
    sh_sum = sc_sum = 0.0
    n_items = 0

    try:
        for batch in test_loaders[load]:
            stim, resp, seq_len, list_length = move_batch(batch)
            out = model(stim, seq_len)

            B = stim.shape[0]
            idx = response_indices(seq_len, list_length)
            bidx = torch.arange(B, device=DEVICE)[:, None]

            h = out["h"][bidx, idx]
            c = out["c"][bidx, idx]
            lh = out["logits_h"][bidx, idx]
            lc = out["logits_c"][bidx, idx]
            y = resp[bidx, idx]

            clean = lh + lc
            clean_correct += (clean.argmax(-1) == y).sum().item()

            pi = _derangement(B, g, DEVICE)

            # Dh: break trial identity for the RECURRENT readout only.
            # The recurrent readout is the official VSR head applied to h.
            hp = model.backbone.vsr_task_head(h[pi]) + lc
            # Dc: break trial identity for the RETRIEVAL readout only.
            cp = lh + model.beta * model.context_head(model.context_mlp(c[pi]))

            hp_correct += (hp.argmax(-1) == y).sum().item()
            cp_correct += (cp.argmax(-1) == y).sum().item()
            total += y.numel()

            # contribution share, averaged over items (not over batches)
            sh_sum += lh.std(dim=-1).sum().item()
            sc_sum += lc.std(dim=-1).sum().item()
            n_items += y.numel()
    finally:
        if was_training:
            model.train()

    a = clean_correct / total
    ah = hp_correct / total
    ac = cp_correct / total
    denom = max(a - CHANCE, 1e-9)
    sh, sc = sh_sum / n_items, sc_sum / n_items

    return {
        "accuracy": a,
        "Dh": float(np.clip((a - ah) / denom, 0, 1)),
        "Dc": float(np.clip((a - ac) / denom, 0, 1)),
        "retrieval_share": float(sc / (sh + sc + 1e-9)),
        "acc_h_resampled": ah,
        "acc_c_resampled": ac,
    }

In [ ]:
# 8b) STRICT SANITY CHECK -- the hybrid's recurrent branch must BE the official model
#
# This is the gate the previous run skipped. If any assertion here fails,
# nothing downstream is interpretable.

_check = build_hybrid(0, beta=1.0)
_check.eval()
official_model.eval()

_max_dl, _max_dh, _max_dfull = 0.0, 0.0, 0.0
with torch.no_grad():
    for _i, _b in enumerate(test_loaders[CFG.operating_load]):
        stim, resp, seq_len, list_length = move_batch(_b)
        off_logits, off_h = official_hidden_states(stim, seq_len, model=official_model)
        o = _check(stim, seq_len)
        _max_dl = max(_max_dl, (off_logits - o["logits_h"]).abs().max().item())
        _max_dh = max(_max_dh, (off_h - o["h"]).abs().max().item())
        _max_dfull = max(_max_dfull, (off_logits - o["logits"]).abs().max().item())
        if _i >= 4:
            break

print("max |official logits - hybrid recurrent logits| :", _max_dl)
print("max |official hidden - hybrid hidden|           :", _max_dh)
print("max |official logits - hybrid FULL logits|      :", _max_dfull)

assert _max_dl == 0.0, "recurrent branch is not the official forward pass"
assert _max_dh == 0.0, "hidden states differ from the official forward pass"
assert _max_dfull == 0.0, "zero-init retrieval head is contributing at step 0"

# Dh permutes h at the INPUT TO THE RECURRENT READOUT, which assumes the
# recurrent readout really is vsr_task_head applied to the GRU state. Verify.
with torch.no_grad():
    _b = next(iter(test_loaders[CFG.operating_load]))
    stim, resp, seq_len, list_length = move_batch(_b)
    o = _check(stim, seq_len)
    _B = stim.shape[0]
    _idx = response_indices(seq_len, list_length)
    _bidx = torch.arange(_B, device=DEVICE)[:, None]
    _d_head = (
        _check.backbone.vsr_task_head(o["h"][_bidx, _idx]) - o["logits_h"][_bidx, _idx]
    ).abs().max().item()
print("max |vsr_task_head(h) - logits_h| at response steps :", _d_head)
assert _d_head < 1e-5, (
    "the recurrent readout is not vsr_task_head(h) -- the official forward pass "
    "transforms h before the head, so Dh must intervene inside that transform"
)

_acc_off = official_recurrent_accuracy(official_model, CFG.operating_load)
_acc_hyb = route_accuracy(_check, CFG.operating_load, route="recurrent")
print(f"official L{CFG.operating_load} accuracy        : {_acc_off:.4f}")
print(f"hybrid recurrent L{CFG.operating_load} accuracy : {_acc_hyb:.4f}")

assert abs(_acc_off - _acc_hyb) < 1e-9, "recurrent accuracy does not reproduce"
assert _acc_hyb >= CFG.viability_threshold, (
    f"recurrent route {_acc_hyb:.4f} below viability threshold"
)
print("PASS: hybrid recurrent branch reproduces the official WorM model exactly")
del _check

# Phase A — retrieval viability and baseline calibration

The recurrent route is already independently viable.

Now we ask whether the **new retrieval route** can also solve exact VSR load 3 while the official backbone is frozen.

Then, without using human curves, choose the smallest fixed `beta` for which the additive hybrid:
- keeps accuracy ≥ 0.95;
- is retrieval-dominant (`Dc > Dh`).

If either condition fails, stop.

In [ ]:
# 11) Pilot retrieval route
PILOT_SEED = 991

pilot = build_hybrid(PILOT_SEED, beta=1.0)

# guard: the recurrent route must still be intact BEFORE retrieval training
_pre = route_accuracy(pilot, CFG.operating_load, route="recurrent")
assert _pre >= CFG.viability_threshold, f"recurrent route broken at build: {_pre:.4f}"

train_retrieval_only(pilot, CFG.retrieval_steps, PILOT_SEED)

retr_acc = route_accuracy(pilot, CFG.operating_load, route="retrieval_raw")
rec_acc = route_accuracy(pilot, CFG.operating_load, route="recurrent")

print("recurrent-only @ L3:", round(rec_acc, 4), "(was", round(_pre, 4), "before)")
print("retrieval-only @ L3:", round(retr_acc, 4))

# The frozen backbone must not have moved at all during retrieval-only training.
assert abs(rec_acc - _pre) < 1e-9, (
    "the frozen recurrent route changed during retrieval-only training -- "
    "check that norm-layer running statistics stayed in eval mode"
)

if retr_acc < CFG.viability_threshold:
    raise RuntimeError(
        f"STOP: retrieval route is not independently viable at L3 "
        f"({retr_acc:.4f} < {CFG.viability_threshold})."
    )

In [ ]:
# 12) Model-only beta calibration
beta_rows = []

for beta in CFG.beta_grid:
    pilot.set_beta(beta)
    met = causal_metrics(pilot, CFG.operating_load)
    beta_rows.append({"beta": beta, **met})
    print(f"beta={beta:g} acc={met['accuracy']:.3f} "
          f"Dh={met['Dh']:.3f} Dc={met['Dc']:.3f} "
          f"share={met['retrieval_share']:.3f}")

BETA_DF = pd.DataFrame(beta_rows)
display(BETA_DF.round(4))

eligible = BETA_DF[
    (BETA_DF["accuracy"] >= CFG.clean_threshold) &
    (BETA_DF["Dc"] > BETA_DF["Dh"])
].sort_values("beta")

if not len(eligible):
    raise RuntimeError(
        "STOP: no fixed beta produced an accurate retrieval-dominant additive hybrid."
    )

CHOSEN_BETA = float(eligible.iloc[0]["beta"])
pilot.set_beta(CHOSEN_BETA)
print("CHOSEN_BETA =", CHOSEN_BETA)

# Phase B — model-only λ pilot

Apply a post-convergence price on the **scaled retrieval logits** and select the smallest λ that:
- preserves clean L3 accuracy ≥ 0.95;
- produces causal inversion (`Dh > Dc`).

Human curves are still not loaded.

In [ ]:
# 13) Joint steering
def set_all_trainable(model):
    for p in model.parameters():
        p.requires_grad = True

def train_mode_joint(model):
    model.train()
    if CFG.freeze_norm_stats:
        # Keep norm running statistics fixed so the only thing that moves is
        # what the price moves. Otherwise the `never` arm is not a null control.
        freeze_norm_(model)

def train_joint(model, steps, lam, seed, log_every=None):
    set_all_trainable(model)

    torch.manual_seed(int(seed))
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(int(seed))

    opt = torch.optim.AdamW(model.parameters(), lr=CFG.lr_steer, weight_decay=0.0)
    it = infinite_loader(make_train_loader(seed))
    hist = []

    for step in range(1, steps + 1):
        train_mode_joint(model)

        stim, resp, seq_len, list_length = move_batch(next(it))
        out = model(stim, seq_len)

        mask = response_mask_general(seq_len, list_length, resp.shape[1])
        loss = F.cross_entropy(out["logits"][mask], resp[mask])

        if lam > 0:
            priced = out["logits_c"][mask]
            loss = loss + lam * priced.norm(dim=-1).mean()

        opt.zero_grad(set_to_none=True)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), CFG.grad_clip)
        opt.step()

        if log_every and (step == 1 or step % log_every == 0):
            met = causal_metrics(model, CFG.operating_load)
            hist.append({"step": step, "loss": float(loss.item()), **met})

    model.eval()
    return pd.DataFrame(hist)

In [ ]:
# 14) Lambda pilot
pilot.set_beta(CHOSEN_BETA)
pilot_start = copy.deepcopy(pilot)

lambda_rows = []

for i, lam in enumerate(CFG.lambda_grid):
    m = copy.deepcopy(pilot_start)

    train_joint(m, CFG.pilot_steps, lam=lam, seed=50000 + i)

    met = causal_metrics(m, CFG.operating_load)
    lambda_rows.append({"lambda": lam, **met})

    print(f"lambda={lam:g} acc={met['accuracy']:.3f} "
          f"Dh={met['Dh']:.3f} Dc={met['Dc']:.3f} "
          f"share={met['retrieval_share']:.3f}")
    del m

LAMBDA_DF = pd.DataFrame(lambda_rows)
display(LAMBDA_DF.round(4))

eligible = LAMBDA_DF[
    (LAMBDA_DF["accuracy"] >= CFG.clean_threshold) &
    (LAMBDA_DF["Dh"] > LAMBDA_DF["Dc"])
].sort_values("lambda")

if not len(eligible):
    raise RuntimeError(
        "STOP: no lambda achieved accurate causal handover in the model-only pilot."
    )

CHOSEN_LAMBDA = float(eligible.iloc[0]["lambda"])
print("CHOSEN_LAMBDA =", CHOSEN_LAMBDA)

# Phase C — confirmatory paired seeds

Each seed independently trains the new retrieval route on the same fixed official backbone, uses the pre-frozen `beta` and `lambda`, and produces paired:

- `pre`
- `never`
- `late`
- `withdrawn`

In [ ]:
# 15) Confirmatory experiment
endpoint_rows = []
trajectory_rows = []
models_for_human = {}

for seed in CFG.seeds:
    print("\n" + "=" * 70)
    print("SEED", seed)

    base = build_hybrid(seed, beta=CHOSEN_BETA)
    rec_before = route_accuracy(base, CFG.operating_load, "recurrent")
    train_retrieval_only(base, CFG.retrieval_steps, seed)
    rec_after = route_accuracy(base, CFG.operating_load, "recurrent")
    assert abs(rec_before - rec_after) < 1e-9, "frozen backbone drifted"

    retr_acc = route_accuracy(base, CFG.operating_load, "retrieval_raw")
    if retr_acc < CFG.viability_threshold:
        print(f"WARNING seed {seed}: retrieval-only={retr_acc:.4f} below threshold")

    pre = causal_metrics(base, CFG.operating_load)
    pre["retrieval_only_acc"] = retr_acc
    pre["recurrent_only_acc"] = rec_after
    endpoint_rows.append({"seed": seed, "arm": "pre", **pre})
    print("pre:", {k: round(v, 3) for k, v in pre.items()
                   if k in ["accuracy", "Dh", "Dc", "retrieval_share"]})

    never = copy.deepcopy(base)
    h = train_joint(never, CFG.steer_steps, lam=0.0,
                    seed=10000 + seed, log_every=CFG.trajectory_every)
    h["seed"], h["arm"] = seed, "never"
    trajectory_rows.append(h)
    endpoint_rows.append({"seed": seed, "arm": "never",
                          **causal_metrics(never, CFG.operating_load)})

    late = copy.deepcopy(base)
    h = train_joint(late, CFG.steer_steps, lam=CHOSEN_LAMBDA,
                    seed=20000 + seed, log_every=CFG.trajectory_every)
    h["seed"], h["arm"] = seed, "late"
    trajectory_rows.append(h)

    late_met = causal_metrics(late, CFG.operating_load)
    endpoint_rows.append({"seed": seed, "arm": "late", **late_met})
    print("late:", {k: round(v, 3) for k, v in late_met.items()
                    if k in ["accuracy", "Dh", "Dc", "retrieval_share"]})

    withdrawn = copy.deepcopy(late)
    h = train_joint(withdrawn, CFG.withdrawal_steps, lam=0.0,
                    seed=30000 + seed, log_every=CFG.trajectory_every)
    h["seed"], h["arm"] = seed, "withdrawn"
    trajectory_rows.append(h)
    endpoint_rows.append({"seed": seed, "arm": "withdrawn",
                          **causal_metrics(withdrawn, CFG.operating_load)})

    # deletion check: does the model still work with retrieval removed?
    for name, m in [("pre", base), ("never", never), ("late", late)]:
        b0 = m.beta_value
        m.set_beta(0.0)
        acc_del = route_accuracy(m, CFG.operating_load, "full")
        m.set_beta(b0)
        for row in endpoint_rows:
            if row["seed"] == seed and row["arm"] == name:
                row["acc_retrieval_deleted"] = acc_del

    models_for_human[(seed, "pre")] = copy.deepcopy(base).cpu()
    models_for_human[(seed, "late")] = copy.deepcopy(late).cpu()
    models_for_human[(seed, "withdrawn")] = copy.deepcopy(withdrawn).cpu()

    del never, late, withdrawn, base
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    pd.DataFrame(endpoint_rows).to_csv(RESULT_DIR / "endpoints.csv", index=False)

ENDPOINTS = pd.DataFrame(endpoint_rows)
TRAJ = pd.concat(trajectory_rows, ignore_index=True)
TRAJ.to_csv(RESULT_DIR / "trajectories.csv", index=False)

display(
    ENDPOINTS.groupby("arm")[["accuracy", "Dh", "Dc", "retrieval_share"]]
    .agg(["mean", "std"]).round(4)
)

In [ ]:
# 15b) Matched post-hoc attenuation control (paper Sec. 4.2 analogue)
#
# beta was CHOSEN to make the pre model retrieval-dominant, so a reviewer will
# ask whether "late steering" just undoes the beta knob. This control lowers
# beta on the PRE model until its retrieval share matches the LATE arm, with no
# training, and reports what that costs in accuracy and where Dh ends up.

def share_at_beta(model, beta, load):
    b0 = model.beta_value
    model.set_beta(beta)
    met = causal_metrics(model, load)
    model.set_beta(b0)
    return met

posthoc_rows = []
for seed in CFG.seeds:
    late_share = float(
        ENDPOINTS[(ENDPOINTS.seed == seed) & (ENDPOINTS.arm == "late")]
        ["retrieval_share"].iloc[0]
    )
    m = models_for_human[(seed, "pre")].to(DEVICE)

    lo, hi = 1e-6, float(CHOSEN_BETA)
    for _ in range(18):                       # bisect beta to match the share
        mid = math.sqrt(lo * hi)
        s = share_at_beta(m, mid, CFG.operating_load)["retrieval_share"]
        if s > late_share:
            hi = mid
        else:
            lo = mid
    matched_beta = math.sqrt(lo * hi)
    met = share_at_beta(m, matched_beta, CFG.operating_load)
    m.cpu()

    posthoc_rows.append({
        "seed": seed, "matched_beta": matched_beta,
        "target_share": late_share, **met,
    })
    print(f"seed {seed}: beta*={matched_beta:.4g} share={met['retrieval_share']:.3f} "
          f"acc={met['accuracy']:.3f} Dh={met['Dh']:.3f}")

POSTHOC = pd.DataFrame(posthoc_rows)
POSTHOC.to_csv(RESULT_DIR / "posthoc_attenuation.csv", index=False)
display(POSTHOC.round(4))

print("\nIf post-hoc attenuation matches the late arm's share but loses accuracy "
      "and does not raise Dh, the handover is not just a rescaling of beta.")

# Phase D — human benchmark

Only now load the published WorM VSR human serial-position curves.

In [ ]:
# 16) Human VSR reference
HUMAN = {
    3: np.array([7.16/8, 6.8/8, 6.9/8], dtype=float),
    4: np.array([6.35/8, 5.25/8, 4.8/8, 6.05/8], dtype=float),
    5: np.array([5.5/8, 4.3/8, 4.2/8, 3.6/8, 5.0/8], dtype=float),
    6: np.array([5.1/8, 4.25/8, 3.55/8, 3.5/8, 2.9/8, 3.95/8], dtype=float),
}

human_rows = []
for L, vals in HUMAN.items():
    for pos, acc in enumerate(vals, 1):
        human_rows.append({"load": L, "position": pos, "human_accuracy": float(acc)})

HUMAN_DF = pd.DataFrame(human_rows)
display(HUMAN_DF)

In [ ]:
# 17) Serial-position curves
@torch.no_grad()
def serial_curve(model, load):
    model = model.to(DEVICE)
    model.eval()

    L = int(load)
    correct = np.zeros(L)
    total = np.zeros(L)

    for batch in test_loaders[L]:
        stim, resp, seq_len, list_length = move_batch(batch)
        out = model(stim, seq_len)

        B = stim.shape[0]
        idx = response_indices(seq_len, list_length)
        bidx = torch.arange(B, device=DEVICE)[:, None]

        pred = out["logits"][bidx, idx].argmax(-1)
        y = resp[bidx, idx]

        for p in range(L):
            correct[p] += (pred[:, p] == y[:, p]).sum().item()
            total[p] += B

    model.cpu()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    return pd.DataFrame({
        "load": L,
        "position": np.arange(1, L + 1),
        "accuracy": correct / total,
    })

curve_rows = []
route_rows = []

for (seed, arm), model in models_for_human.items():
    for L in CFG.human_loads:
        c = serial_curve(model, L)
        c["seed"], c["arm"] = seed, arm
        curve_rows.append(c)

    # The retrieval route was trained only on loads 2-3, so loads 4-6 are OOD
    # for it. Log the per-route accuracies so that confound is visible rather
    # than buried inside the full-hybrid curve.
    model.to(DEVICE)
    for L in CFG.human_loads:
        route_rows.append({
            "seed": seed, "arm": arm, "load": L,
            "acc_full": route_accuracy(model, L, "full"),
            "acc_recurrent": route_accuracy(model, L, "recurrent"),
            "acc_retrieval_raw": route_accuracy(model, L, "retrieval_raw"),
        })
    model.cpu()

CURVES = pd.concat(curve_rows, ignore_index=True)
CURVES.to_csv(RESULT_DIR / "human_serial_curves.csv", index=False)

ROUTE_BY_LOAD = pd.DataFrame(route_rows)
ROUTE_BY_LOAD.to_csv(RESULT_DIR / "route_accuracy_by_load.csv", index=False)
display(
    ROUTE_BY_LOAD.groupby(["arm", "load"])
    [["acc_full", "acc_recurrent", "acc_retrieval_raw"]].mean().round(4)
)

In [ ]:
# 18) Human-distance metrics
def human_distance(g):
    m = g[["load","position","accuracy"]].merge(
        HUMAN_DF, on=["load","position"], how="inner"
    )

    raw = np.sqrt(np.mean((m.accuracy - m.human_accuracy)**2))

    shape_sq = []
    for L, x in m.groupby("load"):
        a = x.accuracy.to_numpy()
        h = x.human_accuracy.to_numpy()
        shape_sq.extend(((a-a.mean()) - (h-h.mean()))**2)
    shape = np.sqrt(np.mean(shape_sq))

    ma = m.groupby("load").accuracy.mean()
    mh = m.groupby("load").human_accuracy.mean()
    loads = sorted(set(ma.index) & set(mh.index))
    load_rmse = np.sqrt(np.mean([(ma[L]-mh[L])**2 for L in loads]))

    r = stats.pearsonr(m.accuracy, m.human_accuracy).statistic if m.accuracy.std() > 1e-8 else np.nan

    return {
        "raw_rmse": float(raw),
        "shape_rmse": float(shape),
        "load_curve_rmse": float(load_rmse),
        "pearson_r": float(r),
    }

dist_rows = []
for (seed, arm), g in CURVES.groupby(["seed","arm"]):
    dist_rows.append({"seed":seed, "arm":arm, **human_distance(g)})

DIST = pd.DataFrame(dist_rows)
display(
    DIST.groupby("arm")[["raw_rmse","shape_rmse","load_curve_rmse","pearson_r"]]
    .agg(["mean","std"]).round(4)
)

def paired(metric, a, b):
    p = DIST.pivot(index="seed", columns="arm", values=metric).dropna()
    d = p[b]-p[a]
    n = len(d)
    mean = d.mean()
    se = d.std(ddof=1)/np.sqrt(n)
    tc = stats.t.ppf(.975, n-1)
    return {
        "metric":metric,
        "comparison":f"{b} - {a}",
        "mean_change":mean,
        "ci95_low":mean-tc*se,
        "ci95_high":mean+tc*se,
        "p":stats.ttest_rel(p[b],p[a]).pvalue,
    }

rows=[]
for metric in ["raw_rmse","shape_rmse","load_curve_rmse"]:
    rows += [
        paired(metric,"pre","late"),
        paired(metric,"late","withdrawn"),
    ]

PAIRED = pd.DataFrame(rows)
display(PAIRED.round(5))
PAIRED.to_csv(RESULT_DIR/"paired_human_tests.csv", index=False)

print("For RMSE: negative late-pre means handover moved closer to humans.")

In [ ]:
# 19) Compact final readout
print("OPERATING LOAD:", CFG.operating_load)
print("N_CLASSES:", N_CLASSES, "| chance:", round(CHANCE, 4))
print("CHOSEN_BETA:", CHOSEN_BETA)
print("CHOSEN_LAMBDA:", CHOSEN_LAMBDA)

print("\nCAUSAL ENDPOINTS")
display(
    ENDPOINTS.groupby("arm")[["accuracy", "Dh", "Dc", "retrieval_share"]]
    .agg(["mean", "std"]).round(4)
)

if "acc_retrieval_deleted" in ENDPOINTS:
    print("\nACCURACY WITH RETRIEVAL DELETED (beta = 0)")
    display(
        ENDPOINTS.dropna(subset=["acc_retrieval_deleted"])
        .groupby("arm")["acc_retrieval_deleted"].agg(["mean", "std"]).round(4)
    )

print("\nMATCHED POST-HOC ATTENUATION CONTROL")
display(POSTHOC[["matched_beta", "accuracy", "Dh", "Dc", "retrieval_share"]]
        .agg(["mean", "std"]).round(4))

print("\nHUMAN DISTANCE")
display(
    DIST.groupby("arm")[["raw_rmse", "shape_rmse", "load_curve_rmse", "pearson_r"]]
    .agg(["mean", "std"]).round(4)
)

print("\nSaved to:", RESULT_DIR)

## Safe claim if positive

> On the exact WorM VSR task, a hybrid built around an independently viable pretrained recurrent backbone can undergo post-convergence causal handover from a retrieval-dominant implementation back to recurrence while preserving task behavior; the handover changes the model's external similarity to published human working-memory curves.

Do **not** claim that humans literally use the recurrent route.

If the retrieval route fails its own viability screen, or if no accurate retrieval-dominant baseline emerges without using human data, stop and report that constraint rather than tuning against the human curves.